### Identify Cell Wall Genes in Non-Arabidopsis Genomes With HMMR3
Grady King

The .xlsx file comes from a literature review in [this paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC10762515/#kiad267-B46) that found all genes in Arabidopsis that relate to cell wall biosynthesis. 

my goal is to build a model that takes these genes, builds a "profile" of how these proteins look, then I can use the model to predict which genes in a different genome fit with the profile, which are likely also related to cell walls. this file is the laziest possible way of doing that, using the off-the-shelf pyhmmer that calls HMMER3, then I can build from there.

In [40]:
import pyhmmer
import pandas as pd
import numpy as np

In [7]:
table_s8 = pd.read_excel("kiad267_supplementary_data.xlsx", sheet_name="Supplemental Table S8", header=1)

In [8]:
print(table_s8)

     Arabidopsis thaliana gene ID (TAIR assembly) Cell wall gene function  \
0                                       At1g05570                    CALS   
1                                       At2g36850                    CALS   
2                                       At4g04970                    CALS   
3                                       At4g03550                    CALS   
4                                       At2g31960                    CALS   
...                                           ...                     ...   
1308                                    At1g62700                    VND5   
1309                                    At5g62380                    VND6   
1310                                    At1g71930                    VND7   
1311                                    At2g44745                  WRKY12   
1312                                    At5g64530                    XND1   

      Arabidopsis thaliana chromosome (TAIR assembly) Start (bp, BC_GFF)  \

The Supplemental Table S8 contains each arabidopsis gene related to cell wall biosynthesis, along with basic information about it, which pfam protein families it belongs to, and its sequence. 

Let's take the pfam values at face value and download all of those profile HMMs. Then, we could say that if a given protein from a new genome matches one of the profiles from arabidopsis, then it must be a cell wall gene. Exactly where the cutoff will be can be decided later.

In [43]:
# isolate pfam IDs so I can download them all

split_pfam = set(table_s8["PFAM (Hmmsearch, E1)"].str.split(';').explode())
split_pfam.remove('')
split_pfam.remove(np.nan)
split_pfam

{'PF00004',
 'PF00005',
 'PF00009',
 'PF00025',
 'PF00046',
 'PF00067',
 'PF00069',
 'PF00071',
 'PF00106',
 'PF00107',
 'PF00141',
 'PF00150',
 'PF00168',
 'PF00169',
 'PF00182',
 'PF00201',
 'PF00221',
 'PF00225',
 'PF00232',
 'PF00235',
 'PF00249',
 'PF00294',
 'PF00295',
 'PF00326',
 'PF00331',
 'PF00332',
 'PF00337',
 'PF00342',
 'PF00349',
 'PF00350',
 'PF00394',
 'PF00408',
 'PF00501',
 'PF00514',
 'PF00534',
 'PF00535',
 'PF00544',
 'PF00553',
 'PF00560',
 'PF00561',
 'PF00642',
 'PF00652',
 'PF00722',
 'PF00735',
 'PF00759',
 'PF00777',
 'PF00847',
 'PF00862',
 'PF00891',
 'PF00892',
 'PF00933',
 'PF00984',
 'PF01031',
 'PF01055',
 'PF01061',
 'PF01073',
 'PF01095',
 'PF01135',
 'PF01190',
 'PF01262',
 'PF01301',
 'PF01357',
 'PF01370',
 'PF01501',
 'PF01565',
 'PF01596',
 'PF01636',
 'PF01697',
 'PF01704',
 'PF01738',
 'PF01762',
 'PF01915',
 'PF01926',
 'PF02018',
 'PF02065',
 'PF02129',
 'PF02140',
 'PF02183',
 'PF02212',
 'PF02263',
 'PF02318',
 'PF02319',
 'PF02364',
 'PF

In [62]:
# download all hmm files using a for loop, requesting from interpro (the database behind pfam)
import requests
import os
import time
from tqdm import tqdm

os.makedirs("hmms", exist_ok = True)

for pfam_id in tqdm(list(split_pfam)):
    if not os.path.exists(f"hmms/{pfam_id}.hmm.gz"):
        with open(f"hmms/{pfam_id}.hmm.gz", "wb+") as f:
            with requests.get(f"https://www.ebi.ac.uk/interpro/api/entry/pfam/{pfam_id}?annotation=hmm") as r:
                f.write(r.content)
        time.sleep(0.25)

100%|██████████| 292/292 [02:25<00:00,  2.01it/s] 
